# 温控设备参数变更审批系统数据分析

读取项目 SQLite 数据库，依次完成数据质量检查、格式规范、描述性统计、异常检测、审批状态分析和趋势可视化。

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

conn = sqlite3.connect('../database/temperature.db')
readings = pd.read_sql_query(
    'SELECT * FROM temperature_readings WHERE deleted_at IS NULL', conn
)
requests = pd.read_sql_query(
    'SELECT * FROM parameter_change_requests WHERE deleted_at IS NULL', conn
)
alarms = pd.read_sql_query(
    'SELECT * FROM alarm_events WHERE deleted_at IS NULL', conn
)
readings.head()

## 1. 数据清洗与格式规范

先检查缺失值和重复值，再将时间字段转换为 `datetime`，将温度字段转换为数值。当前演示数据没有必填字段缺失和完全重复记录，因此不进行人为填补或删除，仅保留可复现的检查过程。

In [ ]:
quality_report = pd.DataFrame({
    'missing': readings[['device_id', 'temperature', 'target_temperature', 'status', 'recorded_at']].isna().sum(),
})
duplicate_total = readings.duplicated(
    subset=['device_id', 'point_id', 'temperature', 'target_temperature', 'recorded_at']
).sum()
readings['recorded_at'] = pd.to_datetime(readings['recorded_at'], errors='coerce')
readings['temperature'] = pd.to_numeric(readings['temperature'], errors='coerce')
readings['target_temperature'] = pd.to_numeric(readings['target_temperature'], errors='coerce')
quality_report, duplicate_total

## 2. 描述性统计与偏差指标

不同设备目标温度不同，因此使用实际温度与目标温度的差值和绝对偏差评价运行稳定性。

In [ ]:
readings['deviation'] = readings['temperature'] - readings['target_temperature']
readings['absolute_deviation'] = readings['deviation'].abs()
summary = readings[['temperature', 'target_temperature', 'deviation', 'absolute_deviation']].describe()
device_summary = readings.groupby('device_id').agg(
    reading_total=('id', 'count'),
    avg_temperature=('temperature', 'mean'),
    avg_absolute_deviation=('absolute_deviation', 'mean'),
    max_absolute_deviation=('absolute_deviation', 'max'),
    abnormal_total=('status', lambda values: (values != 'normal').sum()),
).round(2)
summary, device_summary

## 3. 阈值异常检测

将绝对偏差不小于 6℃ 的读数标记为异常候选，并按偏差从高到低排序。该规则用于课程项目演示，真实生产环境应结合设备、测点和工艺阶段设置动态阈值。

In [ ]:
anomalies = readings.loc[
    readings['absolute_deviation'] >= 6,
    ['device_id', 'temperature', 'target_temperature', 'absolute_deviation', 'recorded_at']
].sort_values('absolute_deviation', ascending=False)
anomalies.head(10), len(anomalies)

## 4. 审批与报警统计

In [ ]:
request_status = requests['status'].value_counts()
parameter_counts = requests['parameter_name'].value_counts()
alarm_summary = alarms.groupby(['severity', 'status']).size().rename('total')
request_status, parameter_counts, alarm_summary

## 5. 趋势与可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sample = readings.sort_values('recorded_at').tail(120)
axes[0].plot(sample['recorded_at'], sample['temperature'], label='temperature')
axes[0].plot(sample['recorded_at'], sample['target_temperature'], label='target')
axes[0].set_title('Temperature trend')
axes[0].legend()
request_status.plot(kind='pie', autopct='%1.1f%%', title='Approval status', ax=axes[1])
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

## 6. 简单移动平均预测

按设备分别计算 12 条读数移动平均，避免将不同目标温度的设备混在同一窗口中。该结果只作为趋势参考，不作为生产控制指令。

In [ ]:
ordered = readings.sort_values(['device_id', 'recorded_at']).copy()
ordered['moving_average_12'] = ordered.groupby('device_id')['temperature'].transform(
    lambda values: values.rolling(12, min_periods=1).mean()
)
ordered.groupby('device_id').tail(1)[
    ['device_id', 'temperature', 'moving_average_12', 'recorded_at']
]